### Messages in Autogen v0.4

We can imaging messages as the way agent communicate - text our Friend.

When we communicate with the agents -----> sending a message when it responds ---> it too sends a message

TextMessage ImageMessage ToolMessage

In [1]:
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage, MultiModalMessage
from autogen_core import Image as AGImage

from PIL import Image
from io import BytesIO
import requests

from dotenv import load_dotenv
import os

In [2]:
from azure.keyvault.secrets import SecretClient
from azure.identity import DefaultAzureCredential


load_dotenv()

vault_url = os.getenv("AZURE_VAULT_URL")

credential = DefaultAzureCredential()

client = SecretClient(credential=credential, vault_url=vault_url)

open_api_key = client.get_secret("openai-api-key").value
open_api_version = client.get_secret("openai-api-version").value
openai_deployment = client.get_secret("openai-deployment").value
openai_endpoint = client.get_secret("openai-endpoint").value

In [3]:
model_client = AzureOpenAIChatCompletionClient(
    model="gpt-4o",
    azure_deployment=openai_deployment,
    api_key=open_api_key,
    api_version=open_api_version,
    azure_endpoint=openai_endpoint,
)

Simplest Type of Message - TextMessage

In [4]:
agent = AssistantAgent(
    name = "text_agent",
    model_client=model_client,
    system_message='You are a helpful assistant. answer question accurately'
)

In [5]:
async def test_text_messages():
    text_msg = TextMessage(content="What is captial of USA ?", source='user')
    result = await agent.run(task=text_msg)
    print(result.messages[-1].content)

await test_text_messages()

The capital of the United States is **Washington, D.C.**


d:\All Python\Microsoft Autogen\.venv\Lib\site-packages\autogen_agentchat\agents\_assistant_agent.py:1109: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


Multi Modal Messages

In [6]:
async def test_multi_modal():

    response = requests.get('https://picsum.photos/id/15/200/300') # 23 for the image of folks
    pil_image = Image.open(BytesIO(response.content))
    ag_image = AGImage(pil_image)

    multi_modal_msg = MultiModalMessage(
        content = ['What is in the image?',ag_image],
        source='user'
    )

    result = await agent.run(task=multi_modal_msg)
    print(result.messages[-1].content)


await test_multi_modal()

The image shows a scenic view of a waterfall surrounded by lush greenery and rocky terrain. There is a flow of water cascading down into a pool or stream, with stones and rocks in the foreground. It appears to depict a natural and tranquil outdoor environment.
